In [ ]:
import os
import re
import math
import random
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple

import gudhi
import gudhi.representations
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


GroupID = Tuple[str, int]


@dataclass
class CloudGroup:
    """Container for one point cloud corresponding to a single (name, timestep)."""
    group_id: GroupID
    name: str
    label: str
    timestep: int
    original_data: pd.DataFrame  # columns: x, y, z
    metadata: Dict[str, Any] = field(default_factory=dict)
    persistence_diagrams: Dict[int, Optional[np.ndarray]] = field(default_factory=dict)
    landscapes: Dict[int, Optional[np.ndarray]] = field(default_factory=dict)


class CloudGroupPCA:
    """
    Helper class for PCA-based analysis and visualization of landscape features
    stored in CloudDataManager.

    Design:
    - Does not own raw data
    - Uses CloudDataManager as the source of truth
    - Reuses manager feature extraction logic to avoid duplication
    """

    VALID_KINDS = {"h0", "h1", "combined"}

    def __init__(self, manager: "CloudDataManager") -> None:
        self.manager = manager

        self.feature_matrices: Dict[str, Optional[pd.DataFrame]] = {
            "h0": None,
            "h1": None,
            "combined": None,
        }

        self.scalers: Dict[str, Optional[StandardScaler]] = {
            "h0": None,
            "h1": None,
            "combined": None,
        }

        self.scaled_matrices: Dict[str, Optional[np.ndarray]] = {
            "h0": None,
            "h1": None,
            "combined": None,
        }

        self.pca_models: Dict[str, Optional[PCA]] = {
            "h0": None,
            "h1": None,
            "combined": None,
        }

        self.pca_results: Dict[str, Optional[Dict[str, Any]]] = {
            "h0": None,
            "h1": None,
            "combined": None,
        }

    # -------------------------
    # Internal helpers
    # -------------------------
    def _validate_kind(self, kind: str) -> None:
        if kind not in self.VALID_KINDS:
            raise ValueError(f"kind must be one of {sorted(self.VALID_KINDS)}")

    def _validate_required_dimensions(self) -> None:
        if self.manager._tda_params is None:
            raise ValueError("TDA features have not been computed yet")

        manager_dims = set(self.manager._tda_params["dimensions"])
        required_dims = {0, 1}
        if not required_dims.issubset(manager_dims):
            raise ValueError(
                "CloudGroupPCA requires both H0 and H1 landscapes to be available. "
                f"Manager dimensions are {sorted(manager_dims)}."
            )

    def _select_group_ids(
        self,
        group_ids: Optional[List[GroupID]] = None,
        label: Optional[str] = None,
        name: Optional[str] = None,
    ) -> List[GroupID]:
        """
        Resolve group selection using the manager's existing indexing rules.
        """
        if group_ids is not None:
            resolved = []
            for gid in group_ids:
                self.manager.get_group(gid)
                resolved.append(gid)

            if label is not None:
                self.manager._validate_label(label)
                resolved = [gid for gid in resolved if self.manager.groups[gid].label == label]

            if name is not None:
                resolved = [gid for gid in resolved if self.manager.groups[gid].name == name]

            return sorted(resolved, key=lambda gid: (gid[0], gid[1]))

        ids = self.manager.list_group_ids()

        if label is not None:
            self.manager._validate_label(label)
            ids = [gid for gid in ids if self.manager.groups[gid].label == label]

        if name is not None:
            ids = [gid for gid in ids if self.manager.groups[gid].name == name]

        return ids

    def _subset_columns(self, df: pd.DataFrame, kind: str) -> pd.DataFrame:
        """
        Select H0, H1, or combined columns from the manager-generated feature matrix.
        """
        self._validate_kind(kind)

        if kind == "combined":
            return df.copy()

        prefix = "H0_" if kind == "h0" else "H1_"
        cols = [c for c in df.columns if c.startswith(prefix)]

        if len(cols) == 0:
            raise ValueError(f"No columns found for {kind.upper()} in feature matrix")

        return df[cols].copy()

    # -------------------------
    # Feature preparation
    # -------------------------
    def build_feature_matrices(
        self,
        group_ids: Optional[List[GroupID]] = None,
        label: Optional[str] = None,
        name: Optional[str] = None,
    ) -> Dict[str, pd.DataFrame]:
        """
        Build feature matrices for H0, H1, and combined H0+H1 landscapes.

        Returns
        -------
        Dict[str, pd.DataFrame]
            Keys: 'h0', 'h1', 'combined'
        """
        self._validate_required_dimensions()

        ids = self._select_group_ids(group_ids=group_ids, label=label, name=name)
        if len(ids) == 0:
            raise ValueError("No groups matched the PCA selection criteria")

        base_df = self.manager.get_feature_matrix(
            label=label,
            name=name,
            dimensions=list(self.manager._tda_params["dimensions"]),
        )

        if group_ids is not None:
            base_df = base_df.loc[ids]

        h0_df = self._subset_columns(base_df, "h0")
        h1_df = self._subset_columns(base_df, "h1")
        combined_df = base_df.copy()

        self.feature_matrices["h0"] = h0_df
        self.feature_matrices["h1"] = h1_df
        self.feature_matrices["combined"] = combined_df

        return {
            "h0": h0_df,
            "h1": h1_df,
            "combined": combined_df,
        }

    def prepare_scaled_matrices(
        self,
        group_ids: Optional[List[GroupID]] = None,
        label: Optional[str] = None,
        name: Optional[str] = None,
    ) -> Dict[str, np.ndarray]:
        """
        Build and separately standardize H0, H1, and combined feature matrices.
        """
        matrices = self.build_feature_matrices(
            group_ids=group_ids,
            label=label,
            name=name,
        )

        output: Dict[str, np.ndarray] = {}

        for kind, df in matrices.items():
            scaler = StandardScaler()
            scaled = scaler.fit_transform(df.to_numpy(dtype=float))

            self.scalers[kind] = scaler
            self.scaled_matrices[kind] = scaled
            output[kind] = scaled

        return output

    # -------------------------
    # PCA
    # -------------------------
    def run_pca(
        self,
        n_components: int = 2,
        group_ids: Optional[List[GroupID]] = None,
        label: Optional[str] = None,
        name: Optional[str] = None,
    ) -> Dict[str, Dict[str, Any]]:
        """
        Run PCA separately on scaled H0, H1, and combined feature matrices.
        """
        if n_components <= 0:
            raise ValueError("n_components must be positive")

        self.prepare_scaled_matrices(
            group_ids=group_ids,
            label=label,
            name=name,
        )

        output: Dict[str, Dict[str, Any]] = {}

        for kind in ("h0", "h1", "combined"):
            X = self.scaled_matrices[kind]
            if X is None:
                raise ValueError(f"Scaled matrix for {kind} is missing")

            max_components = min(X.shape[0], X.shape[1])
            if n_components > max_components:
                raise ValueError(
                    f"n_components={n_components} is too large for {kind}. "
                    f"Maximum allowed is {max_components}."
                )

            pca = PCA(n_components=n_components)
            transformed = pca.fit_transform(X)

            result = {
                "transformed": transformed,
                "components": pca.components_,
                "explained_variance": pca.explained_variance_,
                "explained_variance_ratio": pca.explained_variance_ratio_,
                "index": self.feature_matrices[kind].index.copy(),
            }

            self.pca_models[kind] = pca
            self.pca_results[kind] = result
            output[kind] = result

        return output

    # -------------------------
    # PCA plotting
    # -------------------------
    def _ensure_pca_result(
        self,
        kind: str,
        n_components: int = 2,
        group_ids: Optional[List[GroupID]] = None,
        label: Optional[str] = None,
        name: Optional[str] = None,
    ) -> Dict[str, Any]:
        self._validate_kind(kind)

        result = self.pca_results.get(kind)
        if result is None:
            self.run_pca(
                n_components=n_components,
                group_ids=group_ids,
                label=label,
                name=name,
            )
            result = self.pca_results[kind]

        if result is None:
            raise ValueError(f"PCA result for {kind} is unavailable")

        return result

    def plot_pca_scatter(
        self,
        kind: str = "combined",
        n_components: int = 2,
        group_ids: Optional[List[GroupID]] = None,
        label: Optional[str] = None,
        name: Optional[str] = None,
        color_by: str = "label",
        annotate: bool = False,
        figsize: Tuple[int, int] = (8, 6),
    ) -> None:
        """
        Plot PCA scatter using PC1 vs PC2.

        Parameters
        ----------
        kind : str
            One of 'h0', 'h1', 'combined'
        color_by : str
            One of 'label', 'name', 'timestep'
        """
        self._validate_kind(kind)

        if n_components < 2:
            raise ValueError("PCA scatter plot requires n_components >= 2")

        if color_by not in {"label", "name", "timestep"}:
            raise ValueError("color_by must be one of ['label', 'name', 'timestep']")

        result = self._ensure_pca_result(
            kind=kind,
            n_components=n_components,
            group_ids=group_ids,
            label=label,
            name=name,
        )

        coords = result["transformed"]
        if coords.shape[1] < 2:
            raise ValueError(f"PCA result for {kind} does not contain 2 components")

        index = result["index"]
        group_ids_ordered = list(index.to_list())

        metadata_rows = []
        for gid in group_ids_ordered:
            group = self.manager.get_group((gid[0], gid[1]))
            metadata_rows.append(
                {
                    "group_id": group.group_id,
                    "label": group.label,
                    "name": group.name,
                    "timestep": group.timestep,
                }
            )

        plt.figure(figsize=figsize)

        unique_values = sorted({row[color_by] for row in metadata_rows}, key=str)
        for value in unique_values:
            idxs = [i for i, row in enumerate(metadata_rows) if row[color_by] == value]
            x = coords[idxs, 0]
            y = coords[idxs, 1]
            plt.scatter(x, y, label=str(value), alpha=0.8)

            if annotate:
                for i in idxs:
                    text = f"{metadata_rows[i]['name']}, t={metadata_rows[i]['timestep']}"
                    plt.annotate(text, (coords[i, 0], coords[i, 1]), fontsize=8)

        evr = result["explained_variance_ratio"]
        plt.xlabel(f"PC1 ({100 * evr[0]:.2f}% variance)")
        plt.ylabel(f"PC2 ({100 * evr[1]:.2f}% variance)")
        plt.title(f"PCA Scatter Plot: {kind.upper()}")
        plt.legend(title=color_by)
        plt.tight_layout()
        plt.show()

    def plot_pca_h0(self, **kwargs: Any) -> None:
        """Convenience wrapper for H0 PCA scatter plot."""
        self.plot_pca_scatter(kind="h0", **kwargs)

    def plot_pca_h1(self, **kwargs: Any) -> None:
        """Convenience wrapper for H1 PCA scatter plot."""
        self.plot_pca_scatter(kind="h1", **kwargs)

    def plot_pca_combined(self, **kwargs: Any) -> None:
        """Convenience wrapper for combined H0+H1 PCA scatter plot."""
        self.plot_pca_scatter(kind="combined", **kwargs)


class CloudDataManager:
    """
    Manager for point-cloud groups loaded from files.

    Design rules:
    - One file corresponds to one group.
    - group_id = (name, timestep)
    - TDA parameters are fixed per CloudDataManager after first computation.
    - Repeatability check samples 20% of groups and compares stored vs recomputed H1
      landscapes using L2 distance.
    """

    REQUIRED_POINT_COLUMNS = [
        "Center of the object_0",
        "Center of the object_1",
        "Center of the object_2",
    ]

    OPTIONAL_TIMESTEP_COLUMNS = ["timestep", "t"]

    COLUMN_RENAME_MAP = {
        "Center of the object_0": "x",
        "Center of the object_1": "y",
        "Center of the object_2": "z",
    }

    VALID_LABELS = {"control", "epha2a"}

    def __init__(self) -> None:
        self.groups: Dict[GroupID, CloudGroup] = {}
        self._tda_params: Optional[Dict[str, Any]] = None
        self.max_finite_death: Optional[float] = None
        self.pca = CloudGroupPCA(self)

    # -------------------------
    # Basic accessors
    # -------------------------
    def list_group_ids(self) -> List[GroupID]:
        return sorted(self.groups.keys(), key=lambda gid: (gid[0], gid[1]))

    def list_by_label(self, label: str) -> List[GroupID]:
        self._validate_label(label)
        return [gid for gid in self.list_group_ids() if self.groups[gid].label == label]

    def list_by_name(self, name: str) -> List[GroupID]:
        return [gid for gid in self.list_group_ids() if self.groups[gid].name == name]

    def get_group(self, group_id: GroupID) -> CloudGroup:
        if group_id not in self.groups:
            raise KeyError(f"group_id {group_id} not found")
        return self.groups[group_id]

    # -------------------------
    # Validation and parsing
    # -------------------------
    def _validate_label(self, label: str) -> None:
        if label not in self.VALID_LABELS:
            raise ValueError(f"Label '{label}' must be one of {sorted(self.VALID_LABELS)}")

    def _parse_filename(self, filepath: str) -> Dict[str, Any]:
        """
        Expected conventions:
        - label inferred from filename containing 'control' or 'epha2a'
        - timestep optionally present as t=<int>
        - name must appear at the end of filename stem as name=<value>

        Example:
            control t=0 centroids name=stewie.csv
        """
        filename = os.path.basename(filepath)
        stem, _ = os.path.splitext(filename)
        lower = stem.lower()

        if "control" in lower:
            label = "control"
        elif "epha2a" in lower:
            label = "epha2a"
        else:
            raise ValueError(f"Could not infer label from filename: {filename}")

        t_match = re.search(r"\bt\s*=\s*(\d+)\b", stem, flags=re.IGNORECASE)
        file_timestep = int(t_match.group(1)) if t_match else None

        name_match = re.search(r"name=(.+)$", stem, flags=re.IGNORECASE)
        if not name_match:
            raise ValueError(
                f"Could not extract sample name from filename: {filename}. "
                f"Expected filename stem to end with 'name=<sample_name>'."
            )

        name = name_match.group(1).strip()
        if not name:
            raise ValueError(f"Extracted empty sample name from filename: {filename}")

        return {
            "label": label,
            "timestep": file_timestep,
            "name": name,
        }

    def _read_file(self, filepath: str) -> pd.DataFrame:
        if not os.path.isfile(filepath):
            raise FileNotFoundError(f"File not found: {filepath}")

        _, ext = os.path.splitext(filepath)
        if ext.lower() in [".xls", ".xlsx"]:
            return pd.read_excel(filepath)
        return pd.read_csv(filepath)

    def _extract_csv_timestep(self, raw_df: pd.DataFrame, filepath: str) -> Optional[int]:
        available = [col for col in self.OPTIONAL_TIMESTEP_COLUMNS if col in raw_df.columns]
        if not available:
            return None

        t_col = available[0]
        t_series = pd.to_numeric(raw_df[t_col], errors="raise").dropna()

        if len(t_series) == 0:
            return None

        unique_t = np.unique(t_series.astype(int).to_numpy())
        if len(unique_t) != 1:
            raise ValueError(
                f"Expected exactly one timestep in file '{filepath}', found {unique_t.tolist()}"
            )

        return int(unique_t[0])

    def _preprocess_points(self, raw_df: pd.DataFrame, filepath: str) -> pd.DataFrame:
        missing = [c for c in self.REQUIRED_POINT_COLUMNS if c not in raw_df.columns]
        if missing:
            raise ValueError(f"Missing point columns in '{filepath}': {missing}")

        df = raw_df[self.REQUIRED_POINT_COLUMNS].rename(columns=self.COLUMN_RENAME_MAP).copy()

        for col in ["x", "y", "z"]:
            df[col] = pd.to_numeric(df[col], errors="raise")

        if len(df) == 0:
            raise ValueError(f"No points found in file: {filepath}")

        if not np.isfinite(df[["x", "y", "z"]].to_numpy()).all():
            raise ValueError(f"Point coordinates contain NaN or infinite values in file: {filepath}")

        return df[["x", "y", "z"]]

    # -------------------------
    # Loading
    # -------------------------
    def load_group_from_file(self, filepath: str) -> GroupID:
        file_meta = self._parse_filename(filepath)
        raw_df = self._read_file(filepath)

        points_df = self._preprocess_points(raw_df, filepath)
        csv_timestep = self._extract_csv_timestep(raw_df, filepath)
        file_timestep = file_meta["timestep"]

        if file_timestep is not None and csv_timestep is not None:
            if file_timestep != csv_timestep:
                print(
                    f"[WARNING] Timestep mismatch in '{filepath}': "
                    f"filename timestep={file_timestep}, csv timestep={csv_timestep}. "
                    "Using filename timestep."
                )
            timestep = file_timestep
        elif file_timestep is not None:
            timestep = file_timestep
        elif csv_timestep is not None:
            timestep = csv_timestep
        else:
            raise ValueError(
                f"No timestep found for file '{filepath}'. "
                f"Provide t=<int> in filename or a timestep/t column in the file."
            )

        name = file_meta["name"]
        label = file_meta["label"]
        self._validate_label(label)

        group_id = (name, timestep)
        if group_id in self.groups:
            raise ValueError(f"Duplicate group_id detected: {group_id}")

        group = CloudGroup(
            group_id=group_id,
            name=name,
            label=label,
            timestep=timestep,
            original_data=points_df.copy(),
            metadata={
                "source_file": filepath,
                "label": label,
                "name": name,
                "timestep": timestep,
                "filename_timestep": file_timestep,
                "csv_timestep": csv_timestep,
            },
        )

        self.groups[group_id] = group
        return group_id

    def load_groups_from_files(self, filepaths: List[str]) -> None:
        for filepath in filepaths:
            self.load_group_from_file(str(filepath))

    # -------------------------
    # TDA helpers
    # -------------------------
    def _normalize_tda_params(
        self,
        dimensions: Optional[List[int]],
        n_layers: int,
        resolution: int,
        max_edge_length: Optional[float],
    ) -> Dict[str, Any]:
        if dimensions is None:
            dimensions = [0, 1]

        if len(dimensions) == 0:
            raise ValueError("dimensions must not be empty")
        if any(d < 0 for d in dimensions):
            raise ValueError("All homology dimensions must be nonnegative")
        if n_layers <= 0:
            raise ValueError("n_layers must be positive")
        if resolution <= 0:
            raise ValueError("resolution must be positive")

        return {
            "dimensions": tuple(sorted(set(int(d) for d in dimensions))),
            "n_layers": int(n_layers),
            "resolution": int(resolution),
            "max_edge_length": max_edge_length,
        }

    def _set_or_validate_fixed_tda_params(self, params: Dict[str, Any]) -> None:
        if self._tda_params is None:
            self._tda_params = params.copy()
            return

        if self._tda_params != params:
            raise ValueError(
                "This CloudDataManager already has fixed TDA parameters. "
                f"Existing params={self._tda_params}, requested params={params}. "
                "Use a different CloudDataManager for different TDA settings."
            )

    def _compute_persistence_diagrams_for_points(
        self,
        points: np.ndarray,
        dimensions: List[int],
        max_edge_length: Optional[float],
    ) -> Dict[int, Optional[np.ndarray]]:
        if points.ndim != 2 or points.shape[1] != 3:
            raise ValueError(f"Expected point cloud of shape (n_points, 3), got {points.shape}")

        if len(points) == 0:
            raise ValueError("Cannot compute TDA on an empty point cloud")

        if not np.isfinite(points).all():
            raise ValueError("Point cloud contains NaN or infinite values")

        n_unique = len(np.unique(points, axis=0))
        if 0 in dimensions and n_unique < 1:
            raise ValueError("Point cloud has no valid points for H0 computation")
        if 1 in dimensions and n_unique < 3:
            raise ValueError("Point cloud has too few distinct points for H1 computation")

        max_dim = max(dimensions)

        if max_edge_length is None:
            rips = gudhi.RipsComplex(points=points)
        else:
            rips = gudhi.RipsComplex(points=points, max_edge_length=max_edge_length)

        simplex_tree = rips.create_simplex_tree(max_dimension=max_dim + 1)
        simplex_tree.persistence()

        diagrams: Dict[int, Optional[np.ndarray]] = {}

        for dim in dimensions:
            diag = simplex_tree.persistence_intervals_in_dimension(dim)
            diag = np.asarray(diag, dtype=float).reshape(-1, 2)

            if len(diag) == 0:
                diagrams[dim] = None
                continue

            finite_mask = np.isfinite(diag[:, 1])
            finite_diag = diag[finite_mask]

            if len(finite_diag) == 0:
                diagrams[dim] = None
            else:
                diagrams[dim] = finite_diag

        return diagrams

    def _compute_landscapes_from_diagrams(
        self,
        diagrams: Dict[int, Optional[np.ndarray]],
        dimensions: List[int],
        n_layers: int,
        resolution: int,
    ) -> Dict[int, np.ndarray]:
        if self.max_finite_death is None:
            raise ValueError(
                "self.max_finite_death is not set. Compute persistence diagrams first."
            )

        if self.max_finite_death <= 0:
            sample_range = [0.0, 1.0]
        else:
            sample_range = [0.0, float(self.max_finite_death)]

        landscape_transformer = gudhi.representations.Landscape(
            num_landscapes=n_layers,
            resolution=resolution,
            sample_range=sample_range,
        )

        landscapes: Dict[int, np.ndarray] = {}
        zero_vec = np.zeros(n_layers * resolution, dtype=float)

        for dim in dimensions:
            diag = diagrams.get(dim)

            if diag is None or len(diag) == 0:
                landscapes[dim] = zero_vec.copy()
            else:
                landscapes[dim] = landscape_transformer.fit_transform([diag]).reshape(-1)

        return landscapes

    def _get_max_finite_death_from_diagrams(
        self,
        diagrams: Dict[int, Optional[np.ndarray]],
    ) -> Optional[float]:
        max_death = None

        for diag in diagrams.values():
            if diag is None or len(diag) == 0:
                continue

            finite_deaths = diag[:, 1]
            if len(finite_deaths) == 0:
                continue

            local_max = float(np.max(finite_deaths))
            if max_death is None or local_max > max_death:
                max_death = local_max

        return max_death

    # -------------------------
    # TDA computation
    # -------------------------
    def compute_all_tda_features(
        self,
        dimensions: Optional[List[int]] = None,
        n_layers: int = 5,
        resolution: int = 100,
        max_edge_length: Optional[float] = None,
    ) -> None:
        if len(self.groups) == 0:
            raise ValueError("No groups loaded")

        params = self._normalize_tda_params(
            dimensions=dimensions,
            n_layers=n_layers,
            resolution=resolution,
            max_edge_length=max_edge_length,
        )
        self._set_or_validate_fixed_tda_params(params)

        dims = list(params["dimensions"])

        global_max_finite_death = None

        for group_id in self.list_group_ids():
            group = self.groups[group_id]
            points = group.original_data[["x", "y", "z"]].to_numpy(dtype=float, copy=True)

            diagrams = self._compute_persistence_diagrams_for_points(
                points=points,
                dimensions=dims,
                max_edge_length=params["max_edge_length"],
            )

            local_max = self._get_max_finite_death_from_diagrams(diagrams)
            if local_max is not None:
                if global_max_finite_death is None or local_max > global_max_finite_death:
                    global_max_finite_death = local_max

            if 0 in dims and diagrams.get(0) is None:
                print(f"[CHECK FILE] H0 is empty for group {group_id}")

            if 1 in dims and diagrams.get(1) is None:
                print(f"[CHECK FILE] H1 is empty for group {group_id}")

            group.persistence_diagrams = diagrams

        self.max_finite_death = global_max_finite_death

        for group_id in self.list_group_ids():
            group = self.groups[group_id]
            group.landscapes = self._compute_landscapes_from_diagrams(
                diagrams=group.persistence_diagrams,
                dimensions=dims,
                n_layers=params["n_layers"],
                resolution=params["resolution"],
            )

    # -------------------------
    # Feature extraction
    # -------------------------
    def get_combined_landscape(
        self,
        group_id: GroupID,
        dimensions: Optional[List[int]] = None,
    ) -> np.ndarray:
        if self._tda_params is None:
            raise ValueError("TDA features have not been computed yet")

        group = self.get_group(group_id)

        if dimensions is None:
            dimensions = list(self._tda_params["dimensions"])
        else:
            dimensions = sorted(set(int(d) for d in dimensions))

        manager_dims = set(self._tda_params["dimensions"])
        for dim in dimensions:
            if dim not in manager_dims:
                raise ValueError(
                    f"Requested dimension H{dim} was not computed in this manager. "
                    f"Available dimensions: {sorted(manager_dims)}"
                )

        expected_len = self._tda_params["n_layers"] * self._tda_params["resolution"]
        parts: List[np.ndarray] = []

        for dim in dimensions:
            if dim not in group.landscapes:
                raise ValueError(f"Landscape H{dim} missing for group {group_id}")

            ls = group.landscapes[dim]
            if ls is None:
                ls = np.zeros(expected_len, dtype=float)

            if len(ls) != expected_len:
                raise ValueError(
                    f"Landscape length mismatch for group {group_id}, dim={dim}. "
                    f"Expected {expected_len}, got {len(ls)}"
                )

            parts.append(np.asarray(ls, dtype=float))

        return np.concatenate(parts)

    def get_feature_matrix(
        self,
        label: Optional[str] = None,
        name: Optional[str] = None,
        dimensions: Optional[List[int]] = None,
    ) -> pd.DataFrame:
        if self._tda_params is None:
            raise ValueError("TDA features have not been computed yet")

        ids = self.list_group_ids()

        if label is not None:
            self._validate_label(label)
            ids = [gid for gid in ids if self.groups[gid].label == label]

        if name is not None:
            ids = [gid for gid in ids if self.groups[gid].name == name]

        if dimensions is None:
            dimensions = list(self._tda_params["dimensions"])
        else:
            dimensions = sorted(set(int(d) for d in dimensions))

        n_layers = self._tda_params["n_layers"]
        resolution = self._tda_params["resolution"]

        data = [self.get_combined_landscape(gid, dimensions=dimensions) for gid in ids]

        columns = [
            f"H{dim}_L{layer}_B{bin_idx}"
            for dim in dimensions
            for layer in range(n_layers)
            for bin_idx in range(resolution)
        ]

        index = pd.MultiIndex.from_tuples(ids, names=["name", "timestep"])
        return pd.DataFrame(data=data, index=index, columns=columns)

    # -------------------------
    # Plotting
    # -------------------------
    def plot_persistence_diagrams(
        self,
        group_ids: Optional[List[GroupID]] = None,
        label: Optional[str] = None,
        name: Optional[str] = None,
        figsize_per_subplot: Tuple[int, int] = (6, 5),
    ) -> None:
        """
        Plot stored persistence diagrams for selected groups using gudhi plotting.

        H0 and H1 are shown together on the same axes for each group.
        """
        if len(self.groups) == 0:
            raise ValueError("No groups loaded")

        ids = self.list_group_ids()

        if group_ids is not None:
            requested = set(group_ids)
            missing = [gid for gid in requested if gid not in self.groups]
            if missing:
                raise KeyError(f"group_id(s) not found: {missing}")
            ids = [gid for gid in ids if gid in requested]

        if label is not None:
            self._validate_label(label)
            ids = [gid for gid in ids if self.groups[gid].label == label]

        if name is not None:
            ids = [gid for gid in ids if self.groups[gid].name == name]

        if len(ids) == 0:
            raise ValueError("No groups matched the plotting selection criteria")

        n_rows = len(ids)
        fig, axes = plt.subplots(
            n_rows,
            1,
            figsize=(figsize_per_subplot[0], figsize_per_subplot[1] * n_rows),
            squeeze=False,
        )

        for row_idx, gid in enumerate(ids):
            group = self.groups[gid]
            ax = axes[row_idx][0]

            persistence_pairs: List[Tuple[int, Tuple[float, float]]] = []

            for dim in (0, 1):
                diag = group.persistence_diagrams.get(dim)
                if diag is None or len(diag) == 0:
                    continue

                diag = np.asarray(diag, dtype=float)
                for birth, death in diag:
                    persistence_pairs.append((dim, (float(birth), float(death))))

            if len(persistence_pairs) == 0:
                ax.set_title(f"{group.name}, t={group.timestep}\n(no H0/H1 data)")
                ax.set_xlabel("Birth")
                ax.set_ylabel("Death")
                continue

            gudhi.plot_persistence_diagram(persistence_pairs, axes=ax, legend=True)

            leg = ax.get_legend()
            if leg:
                for text in leg.get_texts():
                    text.set_fontsize(12)

            for poly in ax.findobj(plt.Polygon):
                poly.set_visible(False)

            ax.set_title(f"{group.name}, t={group.timestep}: Persistence Diagram")
            ax.set_xlabel("Birth")
            ax.set_ylabel("Death")

        plt.tight_layout()
        plt.show()


    def plot_persistence_landscape(
        self,
        group_ids: Optional[List[GroupID]] = None,
        label: Optional[str] = None,
        name: Optional[str] = None,
        dimension: int = 1,
        figsize_per_subplot: Tuple[int, int] = (7, 4),
    ) -> None:
        """
        Plot persistence landscapes for selected groups using gudhi Landscape.

        By default this plots H1 only.
        """
        if len(self.groups) == 0:
            raise ValueError("No groups loaded")

        if self._tda_params is None:
            raise ValueError("TDA features have not been computed yet")

        if dimension not in self._tda_params["dimensions"]:
            raise ValueError(
                f"Requested dimension H{dimension} was not computed. "
                f"Available dimensions: {sorted(self._tda_params['dimensions'])}"
            )

        ids = self.list_group_ids()

        if group_ids is not None:
            requested = set(group_ids)
            missing = [gid for gid in requested if gid not in self.groups]
            if missing:
                raise KeyError(f"group_id(s) not found: {missing}")
            ids = [gid for gid in ids if gid in requested]

        if label is not None:
            self._validate_label(label)
            ids = [gid for gid in ids if self.groups[gid].label == label]

        if name is not None:
            ids = [gid for gid in ids if self.groups[gid].name == name]

        if len(ids) == 0:
            raise ValueError("No groups matched the plotting selection criteria")

        n_rows = len(ids)
        fig, axes = plt.subplots(
            n_rows,
            1,
            figsize=(figsize_per_subplot[0], figsize_per_subplot[1] * n_rows),
            squeeze=False,
        )

        num_layers = self._tda_params["n_layers"]
        resolution = self._tda_params["resolution"]
        LS = gudhi.representations.Landscape(
            num_landscapes=num_layers,
            resolution=resolution,
        )

        for row_idx, gid in enumerate(ids):
            group = self.groups[gid]
            ax = axes[row_idx][0]

            diag = group.persistence_diagrams.get(dimension)

            if diag is None or len(diag) == 0:
                ax.set_title(f"{group.name}, t={group.timestep}: H{dimension} Landscape\n(no data)")
                ax.set_xlabel("Filtration value")
                ax.set_ylabel("Landscape value")
                continue

            diag = np.asarray(diag, dtype=float)
            lsc = LS.fit_transform([diag]).reshape(num_layers, -1)

            x_min = float(np.min(diag[:, 0]))
            x_max = float(np.max(diag[:, 1]))
            x_vals = np.linspace(x_min, x_max, resolution)

            for i in range(num_layers):
                ax.plot(x_vals, lsc[i], label=f"Layer {i + 1}")

            ax.legend(fontsize=10)
            ax.set_title(f"{group.name}, t={group.timestep}: H{dimension} Landscape")
            ax.set_xlabel("Filtration value")
            ax.set_ylabel("Landscape value")

        plt.tight_layout()
        plt.show()
    # -------------------------
    # PCA wrappers
    # -------------------------
    def prepare_pca_landscape_matrices(
        self,
        group_ids: Optional[List[GroupID]] = None,
        label: Optional[str] = None,
        name: Optional[str] = None,
    ) -> Dict[str, np.ndarray]:
        """
        Prepare separately scaled H0, H1, and combined landscape matrices.
        """
        return self.pca.prepare_scaled_matrices(
            group_ids=group_ids,
            label=label,
            name=name,
        )

    def run_landscape_pca(
        self,
        n_components: int = 2,
        group_ids: Optional[List[GroupID]] = None,
        label: Optional[str] = None,
        name: Optional[str] = None,
    ) -> Dict[str, Dict[str, Any]]:
        """
        Run PCA on H0, H1, and combined landscape matrices.
        """
        return self.pca.run_pca(
            n_components=n_components,
            group_ids=group_ids,
            label=label,
            name=name,
        )

    def plot_pca_h0(self, **kwargs: Any) -> None:
        """Manager wrapper for H0 PCA plot."""
        self.pca.plot_pca_h0(**kwargs)

    def plot_pca_h1(self, **kwargs: Any) -> None:
        """Manager wrapper for H1 PCA plot."""
        self.pca.plot_pca_h1(**kwargs)

    def plot_pca_combined(self, **kwargs: Any) -> None:
        """Manager wrapper for combined H0+H1 PCA plot."""
        self.pca.plot_pca_combined(**kwargs)

    # -------------------------
    # Repeatability check
    # -------------------------
    def check_group_tda_repeatability(
        self,
        sample_fraction: float = 0.2,
        seed: Optional[int] = None,
    ) -> pd.DataFrame:
        """
        Randomly sample a fraction of current groups, recompute TDA using the fixed
        manager parameters, and compare stored vs recomputed H1 landscapes using L2 distance.

        Returns a DataFrame with one row per sampled group.
        """
        if self._tda_params is None:
            raise ValueError("TDA must be computed before repeatability check")

        if len(self.groups) == 0:
            raise ValueError("No groups available for repeatability check")

        if not (0 < sample_fraction <= 1):
            raise ValueError("sample_fraction must be in (0, 1]")

        if 1 not in self._tda_params["dimensions"]:
            raise ValueError("Repeatability check requires H1 to be part of the fixed TDA dimensions")

        if self.max_finite_death is None:
            raise ValueError("self.max_finite_death is not set. Run compute_all_tda_features first.")

        group_ids = self.list_group_ids()
        sample_size = max(1, math.ceil(sample_fraction * len(group_ids)))

        rng = random.Random(seed)
        sampled_ids = rng.sample(group_ids, sample_size)

        rows: List[Dict[str, Any]] = []

        for group_id in sampled_ids:
            group = self.groups[group_id]

            if 1 not in group.landscapes:
                raise ValueError(f"Stored H1 landscape missing for group {group_id}")

            stored_h1 = np.asarray(group.landscapes[1], dtype=float)

            points = group.original_data[["x", "y", "z"]].to_numpy(dtype=float, copy=True)

            recomputed_diagrams = self._compute_persistence_diagrams_for_points(
                points=points,
                dimensions=list(self._tda_params["dimensions"]),
                max_edge_length=self._tda_params["max_edge_length"],
            )

            recomputed_landscapes = self._compute_landscapes_from_diagrams(
                diagrams=recomputed_diagrams,
                dimensions=list(self._tda_params["dimensions"]),
                n_layers=self._tda_params["n_layers"],
                resolution=self._tda_params["resolution"],
            )

            recomputed_h1 = np.asarray(recomputed_landscapes[1], dtype=float)

            if stored_h1.shape != recomputed_h1.shape:
                raise ValueError(
                    f"H1 landscape shape mismatch for group {group_id}: "
                    f"stored={stored_h1.shape}, recomputed={recomputed_h1.shape}"
                )

            l2_diff = float(np.linalg.norm(stored_h1 - recomputed_h1, ord=2))
            stored_norm = float(np.linalg.norm(stored_h1, ord=2))
            recomputed_norm = float(np.linalg.norm(recomputed_h1, ord=2))
            norm_diff = abs(stored_norm - recomputed_norm)
            allclose = np.allclose(stored_h1, recomputed_h1, atol=1e-8)

            print(
                f"[Repeatability] group={group_id} | "
                f"L2(H1 stored - recomputed)={l2_diff:.6e} | "
                f"| ||stored||_2 - ||recomputed||_2 |={norm_diff:.6e}"
            )

            rows.append(
                {
                    "name": group_id[0],
                    "timestep": group_id[1],
                    "group_id": group_id,
                    "label": group.label,
                    "source_file": group.metadata.get("source_file"),
                    "h1_l2_diff": l2_diff,
                    "h1_norm_diff": norm_diff,
                    "stored_h1_norm": stored_norm,
                    "recomputed_h1_norm": recomputed_norm,
                    "allclose": allclose,
                }
            )

        return pd.DataFrame(rows).sort_values(["name", "timestep"]).reset_index(drop=True)
    
    def _set_bold_title(self, ax, title):
            ax.set_title(title, fontweight='bold', fontsize=16)

    def plot_group_tda_summary(
        self, 
        group_id: GroupID, 
        figsize: Tuple[int, int] = (18, 10)
    ) -> None:
        """
        Generates a summary plot for a single group:
        Row 1: 3D Data | H0 Persistence | H0 Landscape
        Row 2: 3D Data | H1 Persistence | H1 Landscape
        """
        group = self.get_group(group_id)
        if not group.persistence_diagrams:
            raise ValueError(f"TDA features not computed for {group_id}. Run compute_all_tda_features first.")

        fig = plt.figure(figsize=figsize, dpi=150)
        
        # Prepare Data
        points = group.original_data[["x", "y", "z"]].to_numpy()
        # Scale for visualization if needed, or use raw
        scaler = StandardScaler()
        points_scaled = scaler.fit_transform(points)

        tda_params = self._tda_params
        num_layers = tda_params["n_layers"]
        resolution = tda_params["resolution"]
        LS = gudhi.representations.Landscape(num_landscapes=num_layers, resolution=resolution)

        # Iterate through H0 (Row 0) and H1 (Row 1)
        for row, dim in enumerate([0, 1]):
            # --- 1. 3D Plot ---
            ax_3d = fig.add_subplot(2, 3, row * 3 + 1, projection='3d')
            ax_3d.scatter(points_scaled[:, 0], points_scaled[:, 1], points_scaled[:, 2], alpha=0.6, s=10)
            ax_3d.view_init(elev=25, azim=45)
            self._set_bold_title(ax_3d, f"{group.name} (t={group.timestep})\nStandardized 3D Cloud")

            # --- 2. Persistence Diagram ---
            ax_diag = fig.add_subplot(2, 3, row * 3 + 2)
            diag = group.persistence_diagrams.get(dim)
            
            if diag is not None and len(diag) > 0:
                # Format for gudhi plotter: list of (dim, (birth, death))
                pairs = [(dim, (p[0], p[1])) for p in diag]
                gudhi.plot_persistence_diagram(pairs, axes=ax_diag, legend=True)
                
                # Apply snippet styling: Remove gray block
                for poly in ax_diag.findobj(plt.Polygon):
                    poly.set_visible(False)
                
                # Style Legend
                leg = ax_diag.get_legend()
                if leg:
                    for text in leg.get_texts():
                        text.set_fontsize(12)
            
            self._set_bold_title(ax_diag, f"H{dim} Persistence Diagram")

            # --- 3. Landscape Plot ---
            ax_land = fig.add_subplot(2, 3, row * 3 + 3)
            if diag is not None and len(diag) > 0:
                lsc = LS.fit_transform([diag]).reshape(num_layers, -1)
                
                # Calculate x-axis based on birth/death range
                x_min, x_max = diag[:, 0].min(), diag[:, 1].max()
                # Handle cases where all points are same
                if x_min == x_max: x_max += 0.1 
                
                x_vals = np.linspace(x_min, x_max, resolution)
                
                for i in range(num_layers):
                    ax_land.plot(x_vals, lsc[i], label=f'Layer {i+1}')
                ax_land.legend(fontsize=10)
            
            self._set_bold_title(ax_land, f"H{dim} Landscape")

        plt.tight_layout()
        plt.show()
    def plot_comparison_tda(
            self, 
            group_id_1: GroupID, 
            group_id_2: GroupID, 
            share_axes: bool = True,
            figsize: Tuple[int, int] = (20, 12)
        ) -> None:
            """
            Plots two groups side-by-side for direct comparison.
            Rows: H0, H1
            Columns 1-3: Group 1 | Columns 4-6: Group 2
            """
            gids = [group_id_1, group_id_2]
            groups = [self.get_group(gid) for gid in gids]
            
            for g in groups:
                if not g.persistence_diagrams:
                    raise ValueError(f"TDA not computed for {g.group_id}")

            fig = plt.figure(figsize=figsize, dpi=150)
            tda_params = self._tda_params
            res = tda_params["resolution"]
            layers = tda_params["n_layers"]
            LS = gudhi.representations.Landscape(num_landscapes=layers, resolution=res)

            # Pre-calculate global limits if sharing axes
            global_max_death = self.max_finite_death if self.max_finite_death else 1.0
            global_max_land = 0.0
            
            if share_axes:
                # Find the max landscape value across both groups and both dimensions
                for g in groups:
                    for dim in [0, 1]:
                        diag = g.persistence_diagrams.get(dim)
                        if diag is not None and len(diag) > 0:
                            lsc = LS.fit_transform([diag])
                            global_max_land = max(global_max_land, np.max(lsc))

            for row_idx, dim in enumerate([0, 1]):
                for col_offset, group in enumerate(groups):
                    base_col = col_offset * 3
                    diag = group.persistence_diagrams.get(dim)
                    
                    # --- 1. 3D Plot ---
                    ax_3d = fig.add_subplot(2, 6, row_idx * 6 + base_col + 1, projection='3d')
                    pts = StandardScaler().fit_transform(group.original_data[["x", "y", "z"]].to_numpy())
                    ax_3d.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=5, alpha=0.5, c='steelblue')
                    ax_3d.view_init(elev=20, azim=45)
                    self._set_bold_title(ax_3d, f"{group.label.upper()}\n{group.name} (H{dim})")

                    # --- 2. Persistence Diagram ---
                    ax_diag = fig.add_subplot(2, 6, row_idx * 6 + base_col + 2)
                    if diag is not None and len(diag) > 0:
                        gudhi.plot_persistence_diagram([(dim, (p[0], p[1])) for p in diag], axes=ax_diag)
                        for poly in ax_diag.findobj(plt.Polygon):
                            poly.set_visible(False)
                        
                        if share_axes:
                            ax_diag.set_xlim(-0.05, global_max_death * 1.1)
                            ax_diag.set_ylim(-0.05, global_max_death * 1.1)
                    self._set_bold_title(ax_diag, f"Diag H{dim}")

                    # --- 3. Landscape ---
                    ax_land = fig.add_subplot(2, 6, row_idx * 6 + base_col + 3)
                    if diag is not None and len(diag) > 0:
                        lsc = LS.fit_transform([diag]).reshape(layers, -1)
                        # We use the birth/death range for the x-axis 
                        # but we can also sync this if desired
                        x_vals = np.linspace(diag[:, 0].min(), diag[:, 1].max(), res)
                        for i in range(layers):
                            ax_land.plot(x_vals, lsc[i], label=f"L{i+1}")
                        
                        if share_axes:
                            ax_land.set_ylim(-0.02, global_max_land * 1.1)
                            # Optionally sync x-axis to the global filtration range
                            ax_land.set_xlim(-0.05, global_max_death * 1.1)
                    
                    self._set_bold_title(ax_land, f"Land H{dim}")

            plt.tight_layout()
            plt.show()

In [ ]:
from pathlib import Path

data_dir = Path("/workspaces/TDA-workspace/data/Point clouds-selected")
# -------------------------
# Setup
# -------------------------
manager = CloudDataManager()

point_cloud_paths = sorted(
    str(f)
    for f in data_dir.iterdir()
    if f.is_file() and not f.name.startswith(".")
)

assert len(point_cloud_paths) > 0, "No data files found"

# -------------------------
# Load data
# -------------------------
manager.load_groups_from_files(point_cloud_paths)

assert len(manager.groups) == len(point_cloud_paths), \
    "Mismatch between loaded groups and files"

# Check group integrity
for gid, group in manager.groups.items():
    assert isinstance(gid, tuple) and len(gid) == 2
    assert group.name is not None
    assert group.label in manager.VALID_LABELS
    assert group.original_data.shape[1] == 3
    assert {"x", "y", "z"} == set(group.original_data.columns)

# -------------------------
# Compute TDA
# -------------------------
manager.compute_all_tda_features(
    dimensions=[0, 1],
    n_layers=5,
    resolution=100,
    max_edge_length=None,
)

assert manager._tda_params is not None
assert manager.max_finite_death is not None

# Check diagrams + landscapes
for gid, group in manager.groups.items():
    # Persistence diagrams
    assert isinstance(group.persistence_diagrams, dict)
    assert 0 in group.persistence_diagrams
    assert 1 in group.persistence_diagrams

    # Landscapes
    assert isinstance(group.landscapes, dict)
    assert 0 in group.landscapes
    assert 1 in group.landscapes

    expected_len = (
        manager._tda_params["n_layers"]
        * manager._tda_params["resolution"]
    )

    for dim in [0, 1]:
        ls = group.landscapes[dim]
        assert ls is not None
        assert len(ls) == expected_len

# -------------------------
# Feature matrix
# -------------------------
df = manager.get_feature_matrix()
assert isinstance(df, pd.DataFrame)
assert df.shape[0] == len(manager.groups)

# -------------------------
# PCA
# -------------------------
scaled = manager.prepare_pca_landscape_matrices()
assert set(scaled.keys()) == {"h0", "h1", "combined"}

for key, mat in scaled.items():
    assert isinstance(mat, np.ndarray)
    assert mat.shape[0] == len(manager.groups)

pca_results = manager.run_landscape_pca(n_components=2)

for key, result in pca_results.items():
    assert "transformed" in result
    assert result["transformed"].shape[1] == 2

# -------------------------
# Plotting (no crash tests)
# -------------------------
try:
    manager.plot_persistence_diagrams()
    manager.plot_persistence_landscape()
    manager.plot_pca_h0()
    manager.plot_pca_h1()
    manager.plot_pca_combined()
except Exception as e:
    raise AssertionError(f"Plotting failed with error: {e}")

# -------------------------
# Repeatability
# -------------------------
df_repeat = manager.check_group_tda_repeatability(sample_fraction=0.2, seed=42)

assert isinstance(df_repeat, pd.DataFrame)
assert len(df_repeat) > 0
assert "h1_l2_diff" in df_repeat.columns

# sanity: recomputation should be very close
assert (df_repeat["h1_l2_diff"] >= 0).all()

print("All functional tests passed.")


# -------------------------
# Run
# -------------------------
# Example:


In [ ]:
# Compare two specific groups side-by-side
manager.plot_comparison_tda(("stewie", 0), ("meg", 0))